In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="/Users/solomon/mnist-cnn/data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="/Users/solomon/mnist-cnn/data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [13]:
#hyperparameters
#these are "adjustable parameters that let you control the model optimization process"
learning_rate = 1e-3
batch_size = 64
epochs = 10


In [ ]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()
#this combines log softmax with negative log likelihood loss

In [14]:
#SGD (stochastic gradient descent) takes data samples individually, or in this case in batches of 64 and computes the loss gradient over those batches to minimize it
#the path is noisy which promotes generalization and reduces the chance of overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [6]:
#train and test loop

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #runs forward prop through the model
        pred = model(X)
        #computes the cross entropy loss of the models prediction vs the label
        loss = loss_fn(pred, y)

        #running back prop
        loss.backward()
        optimizer.step()
        #clears grad and frees memory
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            #weird regex lol, copied from pytorch docs
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    #intentionally disabling the gradient to speed up computation
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [15]:
#running the loop

loss_fn = nn.CrossEntropyLoss()

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.394058  [   64/60000]
loss: 0.491253  [ 6464/60000]
loss: 0.363310  [12864/60000]
loss: 0.480951  [19264/60000]
loss: 0.413946  [25664/60000]
loss: 0.408607  [32064/60000]
loss: 0.368067  [38464/60000]
loss: 0.505488  [44864/60000]
loss: 0.486183  [51264/60000]
loss: 0.538582  [57664/60000]
Test Error: 
 Accuracy: 84.8%, Avg loss: 0.417170 

Epoch 2
-------------------------------
loss: 0.260559  [   64/60000]
loss: 0.354800  [ 6464/60000]
loss: 0.279063  [12864/60000]
loss: 0.395229  [19264/60000]
loss: 0.444439  [25664/60000]
loss: 0.372919  [32064/60000]
loss: 0.311607  [38464/60000]
loss: 0.471776  [44864/60000]
loss: 0.406387  [51264/60000]
loss: 0.459298  [57664/60000]
Test Error: 
 Accuracy: 86.2%, Avg loss: 0.377968 

Epoch 3
-------------------------------
loss: 0.206888  [   64/60000]
loss: 0.322242  [ 6464/60000]
loss: 0.235798  [12864/60000]
loss: 0.321549  [19264/60000]
loss: 0.388777  [25664/60000]
loss: 0.353198  [32064/600